# 01 — Verify the RadBERT classifier loads and runs

Sanity check only: confirms the base encoder, the fine-tuned classifier head, and the label vocabulary all load correctly, before running anything on real report text.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
from report2label.extraction.label_mapper import LabelVocabulary
from report2label.extraction.predictor import LabelPredictor
from report2label.extraction.thresholding import resolve_thresholds
from report2label.models.model_loader import ModelConfig

model_config = ModelConfig.from_yaml(PROJECT_ROOT / "configs" / "model.yaml")
label_vocab = LabelVocabulary.from_yaml(PROJECT_ROOT / "configs" / "labels.yaml")
thresholds = resolve_thresholds(label_vocab.names, model_config.default_threshold)

predictor = LabelPredictor(model_config, label_vocab, thresholds, evidence_config={"enabled": False})
print(f"Loaded {len(label_vocab)} labels on {predictor.device}")

In [ ]:
sample_sentences = [
    "There is a pleural effusion in the right hemithorax.",
    "The lungs are clear without focal consolidation, nodule, or effusion.",
    "Mild cardiomegaly with mediastinal lymphadenopathy.",
]

for sentence in sample_sentences:
    prediction = predictor.predict(sentence, with_evidence=False)
    top = sorted(prediction.probabilities.items(), key=lambda kv: kv[1], reverse=True)[:5]
    print(sentence)
    for name, prob in top:
        print(f"    {name:40s} {prob:.3f}")
    print()